# Figures and summary tables

Run from the project root or `figures/` after the analysis notebooks. Install `requirements.txt` and Poppler for raster exports.

This notebook generates Fig2, FigS1–S3, and Tables 3, S2, and S3 from the saved responses and `analysis_results/` files.

PDF page sizes are 160 × 203.6939 mm for Fig2 and 174 × 207.4333 mm for each FigS1 page. PNG/TIFF exports use RGB at 600 dpi; FigS3 uses 1200 dpi.


In [1]:
from pathlib import Path
import importlib.util
import os
import sys

# Load bundled PDF packages if needed
if any(importlib.util.find_spec(package) is None for package in ("reportlab", "pypdf")):
    runtime_packages = (Path(os.environ.get("USERPROFILE", "~")).expanduser()
                        / ".cache/codex-runtimes/codex-primary-runtime/dependencies/python/Lib/site-packages")
    if runtime_packages.is_dir():
        sys.path.append(str(runtime_packages))
# Locate the project
ROOT = Path.cwd().resolve()
if not (ROOT / "analysis" / "analysis_utils.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "analysis" / "analysis_utils.py").is_file():
    raise RuntimeError("Run from the project root or its figures folder.")
DATA_DIR = ROOT / "llm_outputs"
RESULT_DIR = ROOT / "analysis_results"
FIGURE_DIR = ROOT / "figures"
for module_dir in (ROOT, FIGURE_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from reference_artwork import render_reference_figures, STABILITY_PT
from figure_exports import export_formats

# Font sizes in points
ARROW_PT = 9.4
EXPECTED_ARROW_PT = 13.0
KNOWLEDGE_ESTIMATE_PT = STABILITY_PT


## Literature comparison figures

Fig2 shows mean paired differences, homeless minus stable. Counts use weekly units; proportions and binary outcomes use percentage points. FigS1 shows Spearman correlations pooled across housing conditions.

Bootstrap agreement is the percentage of estimable resamples matching the fixed literature direction. Mixed expectations and relationships with no estimable resamples show `agreement n/a`. Directions and colors use unrounded estimates.

Knowledge-probe estimates and intervals are LLM-reported responses.


In [2]:
figure_paths = render_reference_figures(
    output_dir=FIGURE_DIR,
    data_dir=DATA_DIR,
    result_dir=RESULT_DIR,
    arrow_pt=ARROW_PT,
    expected_arrow_pt=EXPECTED_ARROW_PT,
    knowledge_estimate_pt=KNOWLEDGE_ESTIMATE_PT,
)
for name, path in figure_paths.items():
    print(f"{name}: {path}")


Fig2: C:\Users\yysma\Documents\hcv_llm\figures\Fig2.pdf
FigS1: C:\Users\yysma\Documents\hcv_llm\figures\FigS1.pdf


## Export figures

Export full PDF pages as RGB PNG/TIFF files. FigS1 has two pages with separate `FigS1_page1` and `FigS1_page2` exports. EPS copies are created when `pdftops` is available.


In [3]:
preview_paths = {}
for name, pdf_path in figure_paths.items():
    preview_paths[name] = export_formats(pdf_path, dpi=600)
    print(f"{name}: exported {len(preview_paths[name])} page(s) as 600-dpi RGB PNG/TIFF")


Fig2: exported 1 page(s) as 600-dpi RGB PNG/TIFF
FigS1: exported 2 page(s) as 600-dpi RGB PNG/TIFF


In [4]:
try:
    from IPython.display import Image, display
except ImportError:
    display = None

for name, pages in preview_paths.items():
    for page_number, path in enumerate(pages, 1):
        print(f"{name} - page {page_number}")
        if display is None:
            print(path)
        else:
            display(Image(filename=str(path), width=760))


Fig2 - page 1
<IPython.core.display.Image object>
FigS1 - page 1
<IPython.core.display.Image object>
FigS1 - page 2
<IPython.core.display.Image object>


## CNEP+ figures and summary tables

FigS2 compares matched CNEP+ and LLM correlations. FigS3 shows follow-up timing sensitivity. Tables S2 and S3 report behavioral prompt agreement and knowledge-probe discrepancies.

Both build switches are enabled. Set a switch to `False` to skip its exports. The summary tables require the main, NYC HANES, and prompt-sensitivity analysis results.


In [5]:
BUILD_OTHER_FIGURES = True
if BUILD_OTHER_FIGURES:
    import pandas as pd
    from analysis.analysis_utils import RESULT_DIR
    from analysis.artwork import plot_cnep_correlations, plot_timing_sensitivity
    plot_cnep_correlations(pd.read_csv(RESULT_DIR / "cnep_mad.csv"))
    plot_timing_sensitivity(pd.read_csv(RESULT_DIR / "cnep_mad_tolerance_sweep.csv"))


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
Saved FigS2: 174 x 234 mm, original layout, 600-dpi PNG
Saved FigS3: 174 x 130.0 mm, EPS/PDF and 1200-dpi RGB PNG/TIFF


In [6]:
BUILD_SUMMARY_TABLES = True
if BUILD_SUMMARY_TABLES:
    import pandas as pd
    from analysis.analysis_utils import RESULT_DIR, MODELS, MODEL_LABELS, COLUMNS, build_table3
    from reference_artwork import NAMES

    table3 = build_table3()
    table3.to_csv(RESULT_DIR / "table3.csv", index=False)
    print(table3.to_string(index=False))

    semantic = pd.read_csv(RESULT_DIR / "prompt_semantic.csv")
    measure_order = ["Overdose", "Public_prop", "Recept_sharing", "Partners", "Drug_in", "Drug_out", "Partner_HCV_prop", "Injections", "Alt_route"]
    outcome_labels = {**NAMES, "Drug_in": "Network in-degree", "Drug_out": "Network out-degree",
                      "Partner_HCV_prop": "Partner HCV proportion", "Overdose": "Overdose"}
    table_s2 = semantic.pivot(index="measure", columns="model", values="directional_consistency") * 100
    table_s2 = table_s2.reindex(index=measure_order, columns=MODELS)
    if table_s2.isna().any().any():
        raise ValueError("Missing behavioral prompt-agreement results for Table S2.")
    table_s2 = table_s2.rename(index=outcome_labels, columns=MODEL_LABELS).rename_axis("Outcome")
    table_s2.round(1).to_csv(RESULT_DIR / "table_s2.csv")

    discrepancies = pd.read_csv(RESULT_DIR / "knowledge_prompt_discrepancies.csv")
    knowledge_labels = {**outcome_labels, "HCV_active": "HCV status"}
    table_s3 = discrepancies[["Model", "exposure", "outcome", "direction_orig", "direction_para"]].copy()
    table_s3["Variable pair"] = [
        f"{knowledge_labels[COLUMNS[x]]} / {knowledge_labels[COLUMNS[y]]}"
        for x, y in zip(table_s3.exposure, table_s3.outcome)
    ]
    table_s3["Model"] = table_s3.Model.map(MODEL_LABELS)
    if table_s3.Model.isna().any():
        raise ValueError("Unknown model in knowledge-probe discrepancies for Table S3.")
    table_s3 = table_s3.rename(columns={"direction_orig": "Original", "direction_para": "Paraphrased"})
    table_s3[["Model", "Variable pair", "Original", "Paraphrased"]].to_csv(RESULT_DIR / "table_s3.csv", index=False)


                                   Metric   Claude Sonnet 4.6       DeepSeek v3.2        GPT-OSS-120B
                        Estimable targets       22/28 (78.6%)       18/28 (64.3%)      28/28 (100.0%)
         Conditional directional recovery       18/22 (81.8%)       13/18 (72.2%)       19/28 (67.9%)
       Unconditional directional recovery       18/28 (64.3%)       13/28 (46.4%)       19/28 (67.9%)
Literature-informed directional agreement       26/28 (92.9%)       25/28 (89.3%)       23/28 (82.1%)
                    CNEP+ correlation MAD    0.11 (0.08–0.14)    0.13 (0.10–0.17)    0.12 (0.10–0.16)
                NYC HANES correlation MAD    0.17 (0.11–0.27)    0.15 (0.10–0.26)    0.18 (0.11–0.28)
             NYC HANES HCV risk-score AUC    0.85 (0.76–0.93)    0.83 (0.73–0.91)    0.78 (0.67–0.88)
        Housing-difference sign agreement 1,670/2,016 (82.8%) 1,881/2,016 (93.3%) 1,155/2,016 (57.3%)
        Agreement between prompt versions       31/32 (96.9%)       31/32 (96.9%) 